In [0]:
!pip install kaggle

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
%restart_python

In [0]:
print("🕒 Time travel — VERSION AS OF 0:")
display(spark.sql(
    f"SELECT COUNT(*) AS count FROM {BRONZE_TABLE} VERSION AS OF 0"
))

print(f"\n✅ Bronze ingestion complete — {bronze_count:,} records")
print("▶  Next: run 02_silver_cleaning")

In [0]:
# Databricks Notebook: 01 - Bronze Layer: Raw Ingestion
# E-Commerce Analytics Pipeline | Serverless | Free Tier

import os
import subprocess
from pyspark.sql import functions as F

CATALOG      = "workspace"
SCHEMA       = "ecommerce"
VOLUME       = "ecommerce_data"
VOLUME_PATH  = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"

print(f"Volume path  : {VOLUME_PATH}")
print(f"Bronze table : {BRONZE_TABLE}")

# ── Create schema & volume ───────────────────────────────────
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{SCHEMA}.{VOLUME}")

# ── Install kaggle ───────────────────────────────────────────
subprocess.run(["pip", "install", "kaggle", "--quiet"], check=True)

os.environ["KAGGLE_USERNAME"] = "adithya811"
os.environ["KAGGLE_KEY"]      = "KGAT_0ac82ff8ea1d5e70a84b4de1dbeed3c3"

# ── Download & unzip into Volume ─────────────────────────────
subprocess.run(
    ["kaggle", "datasets", "download",
     "-d", "carrie1/ecommerce-data",
     "-p", VOLUME_PATH, "--force"],
    check=True, env=os.environ.copy()
)

# Find the zip (name may vary)
zip_file = next(
    (f for f in os.listdir(VOLUME_PATH) if f.endswith(".zip")),
    None
)
if zip_file is None:
    raise FileNotFoundError(f"No zip found in {VOLUME_PATH}. Contents: {os.listdir(VOLUME_PATH)}")

subprocess.run(
    ["unzip", "-o", f"{VOLUME_PATH}/{zip_file}", "-d", VOLUME_PATH],
    check=True
)
os.remove(f"{VOLUME_PATH}/{zip_file}")

# ── Discover the actual CSV filename ─────────────────────────
# Walk in case it extracted into a subdirectory
csv_path = None
for root, dirs, files in os.walk(VOLUME_PATH):
    for fname in files:
        if fname.lower().endswith(".csv"):
            csv_path = os.path.join(root, fname)
            break
    if csv_path:
        break

if csv_path is None:
    raise FileNotFoundError(
        f"No CSV found under {VOLUME_PATH}. "
        f"All files: {[os.path.join(r,f) for r,_,fs in os.walk(VOLUME_PATH) for f in fs]}"
    )

# Convert OS path  /Volumes/workspace/...  →  Spark path  /Volumes/workspace/...
# (Spark on Serverless reads /Volumes/ directly — no dbfs: prefix needed)
spark_csv_path = csv_path
size_mb = os.path.getsize(csv_path) / 1024 / 1024
print(f"CSV found : {csv_path}  ({size_mb:.1f} MB)")

# ── Read CSV with Spark ───────────────────────────────────────
raw_df = (
    spark.read
    .option("header",      "true")
    .option("inferSchema", "true")
    .option("encoding",    "ISO-8859-1")
    .csv(spark_csv_path)
)

raw_count = raw_df.count()
print(f"Raw record count: {raw_count:,}")
raw_df.printSchema()
display(raw_df.limit(5))

# ── Add ingestion metadata ────────────────────────────────────
bronze_df = (
    raw_df
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file",         F.lit(csv_path))
    .withColumn("pipeline_version",    F.lit("1.0"))
)

# ── Write Bronze Delta table ─────────────────────────────────
(
    bronze_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(BRONZE_TABLE)
)

spark.sql(f"""
    ALTER TABLE {BRONZE_TABLE} SET TBLPROPERTIES (
        'delta.logRetentionDuration'         = 'interval 30 days',
        'delta.deletedFileRetentionDuration' = 'interval 7 days',
        'comment' = 'Raw ecommerce transactions — Bronze layer'
    )
""")

# ── Validate ─────────────────────────────────────────────────
bronze_count = spark.table(BRONZE_TABLE).count()
print(f"✅ Bronze table: {BRONZE_TABLE}  ({bronze_count:,} records)")

display(spark.sql(f"DESCRIBE HISTORY {BRONZE_TABLE} LIMIT 3"))
display(spark.sql(f"DESCRIBE DETAIL  {BRONZE_TABLE}"))

# ── Profile ──────────────────────────────────────────────────
# Re-read from table each time to avoid stale DataFrame references

print("Null counts:")
_bt = spark.table(BRONZE_TABLE)
display(_bt.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in _bt.columns]))

print("Top 10 countries:")
display(
    spark.table(BRONZE_TABLE)
    .groupBy("Country")
    .agg(F.count("*").alias("txn_count"))
    .orderBy(F.desc("txn_count"))
    .limit(10)
)

print("Date range:")
display(
    spark.table(BRONZE_TABLE)
    .agg(
        F.min("InvoiceDate").alias("earliest"),
        F.max("InvoiceDate").alias("latest")
    )
)

display(spark.sql(f"SELECT COUNT(*) AS count FROM {BRONZE_TABLE} VERSION AS OF 0"))

print(f"\n✅ Bronze ingestion complete — {bronze_count:,} records")
print("▶  Next: run 02_silver_cleaning")

Volume path  : /Volumes/workspace/ecommerce/ecommerce_data
Bronze table : workspace.ecommerce.bronze_transactions



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


Dataset URL: https://www.kaggle.com/datasets/carrie1/ecommerce-data
License(s): unknown


100%|██████████| 7.20M/7.20M [00:00<00:00, 14.6MB/s]



Archive:  /Volumes/workspace/ecommerce/ecommerce_data/ecommerce-data.zip
  inflating: /Volumes/workspace/ecommerce/ecommerce_data/data.csv  
CSV found : /Volumes/workspace/ecommerce/ecommerce_data/data.csv  (43.5 MB)
Raw record count: 541,909
root
 |-- InvoiceNo: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- UnitPrice: double (nullable = true)
 |-- CustomerID: integer (nullable = true)
 |-- Country: string (nullable = true)



...

Country,txn_count
United Kingdom,495478
Germany,9495
France,8557
EIRE,8196
Spain,2533
Netherlands,2371
Belgium,2069
Switzerland,2002
Portugal,1519
Australia,1259


Date range:


earliest,latest
1/10/2011 10:04,9/9/2011 9:52


count
67501979



✅ Bronze ingestion complete — 541,909 records
▶  Next: run 02_silver_cleaning


In [0]:
# Databricks Notebook: 02 - Silver Layer: Cleaning & Transformation
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# INPUT  : workspace.ecommerce.bronze_transactions
# OUTPUT : workspace.ecommerce.silver_transactions
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, LongType, DoubleType

CATALOG      = "workspace"
SCHEMA       = "ecommerce"
BRONZE_TABLE = f"{CATALOG}.{SCHEMA}.bronze_transactions"
SILVER_TABLE = f"{CATALOG}.{SCHEMA}.silver_transactions"

print(f"Source : {BRONZE_TABLE}")
print(f"Target : {SILVER_TABLE}")

# ── Read Bronze (fresh read, no cached variable reuse) ───────
raw_count = spark.table(BRONZE_TABLE).count()
print(f"Bronze record count: {raw_count:,}")

# ── Apply cleaning rules ─────────────────────────────────────
# Rule 1 — Remove cancellations (InvoiceNo starts with 'C')
# Rule 2 — Remove null CustomerID
# Rule 3 — Remove zero/negative Quantity
# Rule 4 — Remove zero/negative UnitPrice
# Rule 5 — Remove null Description

clean_df = (
    spark.table(BRONZE_TABLE)
    .filter(~F.col("InvoiceNo").startswith("C"))
    .filter(F.col("CustomerID").isNotNull())
    .filter(F.col("Quantity")   > 0)
    .filter(F.col("UnitPrice")  > 0)
    .filter(F.col("Description").isNotNull())
)

clean_count = clean_df.count()
removed     = raw_count - clean_count
print(f"\nCleaning summary:")
print(f"  Raw     : {raw_count:,}")
print(f"  Removed : {removed:,}  ({removed / raw_count * 100:.1f}%)")
print(f"  Clean   : {clean_count:,}")

# ── Cast types ───────────────────────────────────────────────
typed_df = (
    clean_df
    .withColumn("CustomerID",  F.col("CustomerID").cast(LongType()))
    .withColumn("Quantity",    F.col("Quantity").cast(IntegerType()))
    .withColumn("UnitPrice",   F.col("UnitPrice").cast(DoubleType()))
    .withColumn("InvoiceDate",
        F.to_timestamp("InvoiceDate", "M/d/yyyy H:mm"))
)

# ── Derive features ──────────────────────────────────────────
silver_df = (
    typed_df
    .withColumn("revenue",
        F.round(F.col("Quantity") * F.col("UnitPrice"), 2))
    .withColumn("invoice_year",      F.year("InvoiceDate"))
    .withColumn("invoice_month",     F.month("InvoiceDate"))
    .withColumn("invoice_day",       F.dayofmonth("InvoiceDate"))
    .withColumn("invoice_hour",      F.hour("InvoiceDate"))
    .withColumn("day_of_week",       F.dayofweek("InvoiceDate"))
    .withColumn("is_weekend",
        F.when(F.dayofweek("InvoiceDate").isin([1, 7]), 1).otherwise(0))
    .withColumn("quarter",           F.quarter("InvoiceDate"))
    .withColumn("year_month",        F.date_format("InvoiceDate", "yyyy-MM"))
    .withColumn("description_clean", F.trim(F.upper(F.col("Description"))))
    .withColumn("is_uk",
        F.when(F.col("Country") == "United Kingdom", 1).otherwise(0))
    .withColumn("silver_processed_at", F.current_timestamp())
    .select(
        "InvoiceNo", "StockCode", "description_clean",
        "Quantity", "InvoiceDate",
        "invoice_year", "invoice_month", "invoice_day",
        "invoice_hour", "day_of_week", "is_weekend",
        "quarter", "year_month",
        "UnitPrice", "revenue",
        "CustomerID", "Country", "is_uk",
        "silver_processed_at"
    )
)

# ── Write Silver Delta table ──────────────────────────────────
(
    silver_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

print(f"\n✅ Silver table written: {SILVER_TABLE}")

# ── OPTIMIZE + ZORDER ────────────────────────────────────────
spark.sql(f"OPTIMIZE {SILVER_TABLE} ZORDER BY (CustomerID, InvoiceDate)")
print("✅ OPTIMIZE + ZORDER complete")

# ── Set table properties ─────────────────────────────────────
spark.sql(f"""
    ALTER TABLE {SILVER_TABLE}
    SET TBLPROPERTIES (
        'delta.enableChangeDataFeed'         = 'true',
        'delta.logRetentionDuration'         = 'interval 30 days',
        'delta.deletedFileRetentionDuration' = 'interval 7 days',
        'comment' = 'Cleaned ecommerce transactions — Silver layer'
    )
""")

# ── Data quality checks (fresh reads, no stale variable) ─────
neg_rev  = spark.table(SILVER_TABLE).filter(F.col("revenue") <= 0).count()
null_cid = spark.table(SILVER_TABLE).filter(F.col("CustomerID").isNull()).count()
print(f"\n✅ Negative revenue rows : {neg_rev}  (expected 0)")
print(f"✅ Null CustomerID rows  : {null_cid} (expected 0)")

# ── Revenue statistics ───────────────────────────────────────
print("\nRevenue statistics:")
display(
    spark.table(SILVER_TABLE).agg(
        F.count("revenue").alias("row_count"),
        F.round(F.min("revenue"), 2).alias("min_revenue"),
        F.round(F.max("revenue"), 2).alias("max_revenue"),
        F.round(F.avg("revenue"), 2).alias("avg_revenue"),
        F.round(F.sum("revenue"), 2).alias("total_gbp")
    )
)

# ── Monthly revenue trend ────────────────────────────────────
print("\nMonthly revenue trend:")
display(
    spark.table(SILVER_TABLE)
    .groupBy("year_month")
    .agg(
        F.round(F.sum("revenue"),        2).alias("monthly_revenue"),
        F.countDistinct("InvoiceNo")      .alias("invoices"),
        F.countDistinct("CustomerID")     .alias("unique_customers")
    )
    .orderBy("year_month")
)

# ── Time travel check ────────────────────────────────────────
display(spark.sql(
    f"SELECT COUNT(*) AS count FROM {SILVER_TABLE} VERSION AS OF 0"
))

silver_count = spark.table(SILVER_TABLE).count()
print(f"\n✅ Silver layer complete — {silver_count:,} records")
print("▶  Next: run 03_gold_feature_engineering")

Source : workspace.ecommerce.bronze_transactions
Target : workspace.ecommerce.silver_transactions
Bronze record count: 541,909

Cleaning summary:
  Raw     : 541,909
  Removed : 144,025  (26.6%)
  Clean   : 397,884

✅ Silver table written: workspace.ecommerce.silver_transactions
✅ OPTIMIZE + ZORDER complete

✅ Negative revenue rows : 4  (expected 0)
✅ Null CustomerID rows  : 0 (expected 0)

Revenue statistics:


row_count,min_revenue,max_revenue,avg_revenue,total_gbp
397884,0.0,168469.6,22.4,8911407.9



Monthly revenue trend:


year_month,monthly_revenue,invoices,unique_customers
2010-12,572713.89,1400,885
2011-01,569445.04,987,741
2011-02,447137.35,997,758
2011-03,595500.76,1321,974
2011-04,469200.36,1149,856
2011-05,678594.56,1555,1056
2011-06,661213.69,1393,991
2011-07,600091.01,1331,949
2011-08,645343.9,1280,935
2011-09,952838.38,1755,1266


count
397884



✅ Silver layer complete — 397,884 records
▶  Next: run 03_gold_feature_engineering


In [0]:
# Databricks Notebook: 03 - Gold Layer: Feature Engineering & RFM
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# INPUT  : workspace.ecommerce.silver_transactions
# OUTPUT : workspace.ecommerce.gold_rfm_features
#          workspace.ecommerce.gold_product_features
#          workspace.ecommerce.gold_monthly_revenue
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.window import Window
from datetime import timedelta

CATALOG          = "workspace"
SCHEMA           = "ecommerce"
SILVER_TABLE     = f"{CATALOG}.{SCHEMA}.silver_transactions"
GOLD_RFM         = f"{CATALOG}.{SCHEMA}.gold_rfm_features"
GOLD_PRODUCT     = f"{CATALOG}.{SCHEMA}.gold_product_features"
GOLD_MONTHLY_REV = f"{CATALOG}.{SCHEMA}.gold_monthly_revenue"

print(f"Source : {SILVER_TABLE}")
print(f"Targets: {GOLD_RFM}, {GOLD_PRODUCT}, {GOLD_MONTHLY_REV}")

# ── Reference date for recency ───────────────────────────────
max_date       = spark.table(SILVER_TABLE).agg(F.max("InvoiceDate")).collect()[0][0]
reference_date = max_date + timedelta(days=1)
print(f"\nRFM reference date : {reference_date}")
print(f"Silver records     : {spark.table(SILVER_TABLE).count():,}")

# ════════════════════════════════════════════════════════════
# GOLD TABLE 1 — RFM FEATURES
# ════════════════════════════════════════════════════════════

# ── Build RFM base aggregation ───────────────────────────────
rfm_base = (
    spark.table(SILVER_TABLE)
    .groupBy("CustomerID")
    .agg(
        F.datediff(F.lit(reference_date), F.max("InvoiceDate"))
            .alias("recency_days"),
        F.countDistinct("InvoiceNo")
            .alias("frequency"),
        F.round(F.sum("revenue"), 2)
            .alias("monetary"),
        F.round(F.avg("revenue"), 2)
            .alias("avg_order_value"),
        F.countDistinct("StockCode")
            .alias("unique_products_bought"),
        F.countDistinct("year_month")
            .alias("active_months"),
        F.sum("Quantity")
            .alias("total_units_bought"),
        F.min("InvoiceDate").alias("first_purchase_date"),
        F.max("InvoiceDate").alias("last_purchase_date"),
        F.first("Country").alias("country"),
        F.first("is_uk").alias("is_uk"),
    )
    .withColumn("customer_lifespan_days",
        F.datediff(
            F.col("last_purchase_date"),
            F.col("first_purchase_date")
        ))
)

# ── Compute quartile thresholds ──────────────────────────────
rec_q = rfm_base.approxQuantile("recency_days", [0.25, 0.5, 0.75], 0.01)
frq_q = rfm_base.approxQuantile("frequency",    [0.25, 0.5, 0.75], 0.01)
mon_q = rfm_base.approxQuantile("monetary",     [0.25, 0.5, 0.75], 0.01)

# ── Score and segment ────────────────────────────────────────
rfm_scored = (
    rfm_base
    .withColumn("R_score",
        F.when(F.col("recency_days") <= rec_q[0], 4)
        .when(F.col("recency_days") <= rec_q[1], 3)
        .when(F.col("recency_days") <= rec_q[2], 2)
        .otherwise(1))
    .withColumn("F_score",
        F.when(F.col("frequency") > frq_q[2], 4)
        .when(F.col("frequency") > frq_q[1], 3)
        .when(F.col("frequency") > frq_q[0], 2)
        .otherwise(1))
    .withColumn("M_score",
        F.when(F.col("monetary") > mon_q[2], 4)
        .when(F.col("monetary") > mon_q[1], 3)
        .when(F.col("monetary") > mon_q[0], 2)
        .otherwise(1))
    .withColumn("RFM_score",
        F.col("R_score") + F.col("F_score") + F.col("M_score"))
    .withColumn("rfm_segment",
        F.when(
            (F.col("R_score") >= 3) & (F.col("F_score") >= 3) & (F.col("M_score") >= 3),
            "Champions")
        .when((F.col("R_score") >= 3) & (F.col("F_score") >= 2), "Loyal Customers")
        .when((F.col("R_score") >= 3) & (F.col("F_score") == 1), "Recent Customers")
        .when((F.col("R_score") == 2) & (F.col("F_score") >= 2), "Potential Loyalists")
        .when((F.col("R_score") == 1) & (F.col("F_score") >= 2), "At-Risk")
        .when(
            (F.col("R_score") == 1) & (F.col("F_score") == 1) & (F.col("M_score") >= 3),
            "Cant Lose Them")
        .when((F.col("R_score") == 1) & (F.col("F_score") == 1), "Lost")
        .otherwise("Needs Attention"))
    .withColumn("churned",
        F.when(F.col("R_score") == 1, 1).otherwise(0))
    .withColumn("gold_created_at", F.current_timestamp())
)

# ── Write Gold RFM ───────────────────────────────────────────
(
    rfm_scored.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_RFM)
)

spark.sql(f"OPTIMIZE {GOLD_RFM} ZORDER BY (CustomerID, RFM_score)")
print(f"\n✅ {GOLD_RFM} — {spark.table(GOLD_RFM).count():,} customers")

# ── RFM segment summary (fresh read, explicit alias) ─────────
print("\nRFM Segment breakdown:")
display(
    spark.table(GOLD_RFM)
    .groupBy("rfm_segment")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("monetary"),     2).alias("avg_spend_gbp"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days")
    )
    .orderBy(F.desc("customers"))
)

# ════════════════════════════════════════════════════════════
# GOLD TABLE 2 — PRODUCT FEATURES
# ════════════════════════════════════════════════════════════

product_df = (
    spark.table(SILVER_TABLE)
    .groupBy("StockCode", "description_clean")
    .agg(
        F.sum("Quantity")             .alias("total_units_sold"),
        F.round(F.sum("revenue"), 2)  .alias("total_revenue"),
        F.round(F.avg("UnitPrice"), 2).alias("avg_unit_price"),
        F.countDistinct("CustomerID") .alias("unique_buyers"),
        F.countDistinct("InvoiceNo")  .alias("invoice_count"),
        F.countDistinct("Country")    .alias("countries_sold_in"),
    )
    .withColumn("revenue_rank",
        F.rank().over(Window.orderBy(F.desc("total_revenue"))))
    .withColumn("popularity_rank",
        F.rank().over(Window.orderBy(F.desc("unique_buyers"))))
    .withColumn("gold_created_at", F.current_timestamp())
)

(
    product_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_PRODUCT)
)

spark.sql(f"OPTIMIZE {GOLD_PRODUCT} ZORDER BY (revenue_rank)")
print(f"✅ {GOLD_PRODUCT} — {spark.table(GOLD_PRODUCT).count():,} products")

print("\nTop 10 products by revenue:")
display(
    spark.table(GOLD_PRODUCT)
    .orderBy("revenue_rank")
    .limit(10)
)

# ════════════════════════════════════════════════════════════
# GOLD TABLE 3 — MONTHLY REVENUE
# ════════════════════════════════════════════════════════════

w_month = Window.orderBy("year_month")

monthly_df = (
    spark.table(SILVER_TABLE)
    .groupBy("year_month", "invoice_year", "invoice_month")
    .agg(
        F.round(F.sum("revenue"),         2).alias("total_revenue"),
        F.countDistinct("InvoiceNo")       .alias("total_invoices"),
        F.countDistinct("CustomerID")      .alias("unique_customers"),
        F.sum("Quantity")                  .alias("total_units"),
        F.round(F.avg("revenue"),         2).alias("avg_order_value"),
        F.countDistinct("StockCode")       .alias("unique_products"),
    )
    .withColumn("prev_revenue",
        F.lag("total_revenue", 1).over(w_month))
    .withColumn("mom_growth_pct",
        F.round(
            (F.col("total_revenue") - F.col("prev_revenue"))
            / F.col("prev_revenue") * 100, 2))
    .withColumn("gold_created_at", F.current_timestamp())
    .orderBy("year_month")
)

(
    monthly_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_MONTHLY_REV)
)

print(f"✅ {GOLD_MONTHLY_REV}")

print("\nMonthly revenue trend:")
display(spark.table(GOLD_MONTHLY_REV).orderBy("year_month"))

# ── Time travel check ────────────────────────────────────────
display(spark.sql(
    f"SELECT COUNT(*) AS count FROM {GOLD_RFM} VERSION AS OF 0"
))

print("\n✅ Gold layer complete")
print("▶  Next: run 04_ml_pipeline")

Source : workspace.ecommerce.silver_transactions
Targets: workspace.ecommerce.gold_rfm_features, workspace.ecommerce.gold_product_features, workspace.ecommerce.gold_monthly_revenue

RFM reference date : 2011-12-10 12:50:00
Silver records     : 397,884

✅ workspace.ecommerce.gold_rfm_features — 4,338 customers

RFM Segment breakdown:


rfm_segment,customers,avg_spend_gbp,avg_recency_days
Champions,1299,4979.89,17.2
Potential Loyalists,676,1394.9,81.8
Lost,674,247.93,263.6
Loyal Customers,488,948.63,21.8
Needs Attention,419,454.52,82.4
At-Risk,382,1100.35,211.3
Recent Customers,348,346.04,26.6
Cant Lose Them,52,2660.36,255.0


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✅ workspace.ecommerce.gold_product_features — 3,894 products

Top 10 products by revenue:


StockCode,description_clean,total_units_sold,total_revenue,avg_unit_price,unique_buyers,invoice_count,countries_sold_in,revenue_rank,popularity_rank,gold_created_at
23843,"PAPER CRAFT , LITTLE BIRDIE",80995,168469.6,2.08,1,1,1,1,3670,2026-03-19T06:48:02.460Z
22423,REGENCY CAKESTAND 3 TIER,12402,142592.95,12.48,881,1703,29,2,1,2026-03-19T06:48:02.460Z
85123A,WHITE HANGING HEART T-LIGHT HOLDER,36725,100448.15,2.89,856,1971,16,3,2,2026-03-19T06:48:02.460Z
85099B,JUMBO BAG RED RETROSPOT,46181,85220.78,2.02,635,1600,20,4,6,2026-03-19T06:48:02.460Z
23166,MEDIUM CERAMIC TOP STORAGE JAR,77916,81416.73,1.22,138,195,10,5,614,2026-03-19T06:48:02.460Z
POST,POSTAGE,3120,77803.96,31.57,331,1099,23,6,89,2026-03-19T06:48:02.460Z
47566,PARTY BUNTING,15291,68844.33,4.88,708,1379,20,7,3,2026-03-19T06:48:02.460Z
84879,ASSORTED COLOUR BIRD ORNAMENT,35362,56580.34,1.68,678,1375,16,8,4,2026-03-19T06:48:02.460Z
M,MANUAL,7173,53779.93,175.29,197,253,11,9,322,2026-03-19T06:48:02.460Z
23084,RABBIT NIGHT LIGHT,27202,51346.2,2.01,450,801,19,10,26,2026-03-19T06:48:02.460Z


✅ workspace.ecommerce.gold_monthly_revenue

Monthly revenue trend:


year_month,invoice_year,invoice_month,total_revenue,total_invoices,unique_customers,total_units,avg_order_value,unique_products,prev_revenue,mom_growth_pct,gold_created_at
2010-12,2010,12,572713.89,1400,885,312265,21.9,2411,null,null,2026-03-19T06:48:09.348Z
2011-01,2011,1,569445.04,987,741,349098,26.82,2121,572713.89,-0.57,2026-03-19T06:48:09.348Z
2011-02,2011,2,447137.35,997,758,265622,22.44,2124,569445.04,-21.48,2026-03-19T06:48:09.348Z
2011-03,2011,3,595500.76,1321,974,348503,21.91,2234,447137.35,33.18,2026-03-19T06:48:09.348Z
2011-04,2011,4,469200.36,1149,856,292222,20.72,2217,595500.76,-21.21,2026-03-19T06:48:09.348Z
2011-05,2011,5,678594.56,1555,1056,373601,23.96,2219,469200.36,44.63,2026-03-19T06:48:09.348Z
2011-06,2011,6,661213.69,1393,991,363699,24.32,2339,678594.56,-2.56,2026-03-19T06:48:09.348Z
2011-07,2011,7,600091.01,1331,949,369420,22.37,2351,661213.69,-9.24,2026-03-19T06:48:09.348Z
2011-08,2011,8,645343.9,1280,935,398121,23.9,2356,600091.01,7.54,2026-03-19T06:48:09.348Z
2011-09,2011,9,952838.38,1755,1266,544897,23.8,2545,645343.9,47.65,2026-03-19T06:48:09.348Z


count
4338



✅ Gold layer complete
▶  Next: run 04_ml_pipeline


In [0]:
# Databricks Notebook: 04 - ML Pipeline: Churn, Segmentation & Forecast
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# INPUT  : workspace.ecommerce.gold_rfm_features
#          workspace.ecommerce.gold_monthly_revenue
# OUTPUT : MLflow experiments + registered models
# ============================================================

import os
import mlflow
import mlflow.spark
from mlflow.models.signature import infer_signature
from pyspark.ml.feature import VectorAssembler, StandardScaler, MinMaxScaler
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.clustering import KMeans
from pyspark.ml.regression import LinearRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
    RegressionEvaluator,
    ClusteringEvaluator,
)
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG    = "workspace"
SCHEMA     = "ecommerce"
VOLUME     = "ecommerce_data"
GOLD_RFM   = f"{CATALOG}.{SCHEMA}.gold_rfm_features"
GOLD_REV   = f"{CATALOG}.{SCHEMA}.gold_monthly_revenue"
EXP_NAME   = "/ecommerce_ml_pipeline"
N_CLUSTERS = 4
TEST_SPLIT = 0.2

# ── Serverless: env var only, no spark.conf.set ──────────────
MLFLOW_TMP = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/mlflow_tmp"
os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP
print(f"MLFLOW_DFS_TMP : {MLFLOW_TMP}")

mlflow.set_experiment(EXP_NAME)
print(f"MLflow experiment: {EXP_NAME}")

# ════════════════════════════════════════════════════════════
# MODEL 1 — CHURN PREDICTION (GBTClassifier)
# ════════════════════════════════════════════════════════════

CHURN_FEATURES = [
    "recency_days", "frequency", "monetary",
    "avg_order_value", "unique_products_bought",
    "active_months", "total_units_bought",
    "customer_lifespan_days", "R_score", "F_score", "M_score",
    "is_uk"
]

churn_df = (
    spark.table(GOLD_RFM)
    .select(CHURN_FEATURES + ["churned"])
    .na.fill(0)
)

train_c, test_c = churn_df.randomSplit([1 - TEST_SPLIT, TEST_SPLIT], seed=42)

churn_total = train_c.count()
churn_pos   = train_c.filter(F.col("churned") == 1).count()
print(f"\nChurn — Train: {churn_total:,} | Test: {test_c.count():,}")
print(f"Churn rate   : {churn_pos / churn_total:.1%}")

with mlflow.start_run(run_name="churn_gbt") as run:

    assembler = VectorAssembler(inputCols=CHURN_FEATURES, outputCol="raw_features")
    scaler    = StandardScaler(
        inputCol="raw_features", outputCol="features",
        withMean=True, withStd=True
    )
    gbt = GBTClassifier(
        labelCol="churned", featuresCol="features",
        maxIter=50, maxDepth=5, stepSize=0.1, seed=42
    )
    pipe = Pipeline(stages=[assembler, scaler, gbt])

    mlflow.log_params({
        "model":      "GBTClassifier",
        "maxIter":    50,
        "maxDepth":   5,
        "stepSize":   0.1,
        "n_features": len(CHURN_FEATURES),
        "train_rows": churn_total,
    })

    churn_model = pipe.fit(train_c)
    preds       = churn_model.transform(test_c)

    auc  = BinaryClassificationEvaluator(labelCol="churned").evaluate(preds)
    f1   = MulticlassClassificationEvaluator(
               labelCol="churned", metricName="f1").evaluate(preds)
    acc  = MulticlassClassificationEvaluator(
               labelCol="churned", metricName="accuracy").evaluate(preds)
    prec = MulticlassClassificationEvaluator(
               labelCol="churned", metricName="weightedPrecision").evaluate(preds)
    rec  = MulticlassClassificationEvaluator(
               labelCol="churned", metricName="weightedRecall").evaluate(preds)

    mlflow.log_metrics({
        "auc_roc":   round(auc,  4),
        "f1_score":  round(f1,   4),
        "accuracy":  round(acc,  4),
        "precision": round(prec, 4),
        "recall":    round(rec,  4),
    })

    # Infer signature from input features + predictions (required for UC)
    input_example  = train_c.select(CHURN_FEATURES).limit(5).toPandas()
    preds_sample   = preds.select("prediction").limit(5).toPandas()
    churn_signature = infer_signature(input_example, preds_sample)

    mlflow.spark.log_model(
        churn_model, "churn_model",
        dfs_tmpdir=MLFLOW_TMP,
        signature=churn_signature,
        input_example=input_example,
        registered_model_name="ecommerce_churn_predictor"
    )

    CHURN_RUN = run.info.run_id
    print(f"\nChurn Results — AUC={auc:.4f}  F1={f1:.4f}  Acc={acc:.4f}")
    print(f"Run ID: {CHURN_RUN}")

# ════════════════════════════════════════════════════════════
# MODEL 2 — CUSTOMER SEGMENTATION (KMeans)
# ════════════════════════════════════════════════════════════

SEG_FEATURES = [
    "recency_days", "frequency", "monetary",
    "avg_order_value", "customer_lifespan_days"
]

seg_df = (
    spark.table(GOLD_RFM)
    .select(["CustomerID"] + SEG_FEATURES)
    .na.fill(0)
)

with mlflow.start_run(run_name="segmentation_kmeans") as run:

    asm_s  = VectorAssembler(inputCols=SEG_FEATURES, outputCol="raw_features")
    scl_s  = MinMaxScaler(inputCol="raw_features", outputCol="features")
    kmeans = KMeans(
        k=N_CLUSTERS, seed=42,
        featuresCol="features", predictionCol="segment_id", maxIter=20
    )
    pipe_s = Pipeline(stages=[asm_s, scl_s, kmeans])

    mlflow.log_params({
        "model":   "KMeans",
        "k":       N_CLUSTERS,
        "maxIter": 20,
    })

    seg_model = pipe_s.fit(seg_df)
    seg_preds = seg_model.transform(seg_df)

    silhouette = ClusteringEvaluator(
        featuresCol="features",
        predictionCol="segment_id",
        metricName="silhouette"
    ).evaluate(seg_preds)

    mlflow.log_metrics({
        "silhouette_score": round(silhouette, 4),
        "n_clusters":       N_CLUSTERS,
    })

    # Infer signature (required for UC)
    seg_input_example  = seg_df.select(SEG_FEATURES).limit(5).toPandas()
    seg_preds_sample   = seg_preds.select("segment_id").limit(5).toPandas()
    seg_signature      = infer_signature(seg_input_example, seg_preds_sample)

    mlflow.spark.log_model(
        seg_model, "segmentation_model",
        dfs_tmpdir=MLFLOW_TMP,
        signature=seg_signature,
        input_example=seg_input_example,
        registered_model_name="ecommerce_customer_segmenter"
    )

    SEG_RUN = run.info.run_id
    print(f"\nSegmentation — Silhouette={silhouette:.4f}  k={N_CLUSTERS}")
    print(f"Run ID: {SEG_RUN}")

display(
    seg_preds.groupBy("segment_id")
    .agg(
        F.count("*").alias("customer_count"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency"),
        F.round(F.avg("frequency"),    1).alias("avg_frequency"),
        F.round(F.avg("monetary"),     2).alias("avg_monetary"),
    )
    .orderBy("segment_id")
)

# ════════════════════════════════════════════════════════════
# MODEL 3 — REVENUE FORECASTING (Linear Regression)
# ════════════════════════════════════════════════════════════

REV_FEATURES = [
    "time_index", "invoice_month", "invoice_year",
    "lag_1_revenue", "lag_2_revenue", "lag_3_revenue",
    "total_invoices", "unique_customers"
]

w_ts = Window.orderBy("year_month")

forecast_df = (
    spark.table(GOLD_REV)
    .orderBy("year_month")
    .withColumn("time_index",    F.row_number().over(w_ts))
    .withColumn("lag_1_revenue", F.lag("total_revenue", 1).over(w_ts))
    .withColumn("lag_2_revenue", F.lag("total_revenue", 2).over(w_ts))
    .withColumn("lag_3_revenue", F.lag("total_revenue", 3).over(w_ts))
    .na.drop()
)

n       = forecast_df.count()
train_r = forecast_df.filter(F.col("time_index") <= n - 2)
test_r  = forecast_df.filter(F.col("time_index") >  n - 2)

with mlflow.start_run(run_name="revenue_forecast_lr") as run:

    asm_r  = VectorAssembler(inputCols=REV_FEATURES, outputCol="features")
    lr     = LinearRegression(
        labelCol="total_revenue", featuresCol="features",
        regParam=0.1, elasticNetParam=0.5
    )
    pipe_r = Pipeline(stages=[asm_r, lr])

    mlflow.log_params({
        "model":           "LinearRegression",
        "regParam":        0.1,
        "elasticNetParam": 0.5,
        "train_months":    train_r.count(),
    })

    rev_model = pipe_r.fit(train_r)
    rev_preds = rev_model.transform(test_r)

    reg  = RegressionEvaluator(
        labelCol="total_revenue", predictionCol="prediction")
    rmse = reg.setMetricName("rmse").evaluate(rev_preds)
    mae  = reg.setMetricName("mae").evaluate(rev_preds)
    r2   = reg.setMetricName("r2").evaluate(rev_preds)

    mape = (
        rev_preds
        .withColumn("ape",
            F.abs(
                (F.col("total_revenue") - F.col("prediction"))
                / F.col("total_revenue")
            ) * 100)
        .agg(F.avg("ape").alias("mape"))
        .collect()[0]["mape"]
    )

    mlflow.log_metrics({
        "rmse": round(rmse, 2),
        "mae":  round(mae,  2),
        "r2":   round(r2,   4),
        "mape": round(mape, 2),
    })

    # Infer signature (required for UC)
    rev_input_example = train_r.select(REV_FEATURES).limit(5).toPandas()
    rev_preds_sample  = rev_preds.select("prediction").limit(5).toPandas()
    rev_signature     = infer_signature(rev_input_example, rev_preds_sample)

    mlflow.spark.log_model(
        rev_model, "revenue_forecast_model",
        dfs_tmpdir=MLFLOW_TMP,
        signature=rev_signature,
        input_example=rev_input_example,
        registered_model_name="ecommerce_revenue_forecaster"
    )

    REV_RUN = run.info.run_id
    print(f"\nRevenue Forecast — RMSE={rmse:,.0f}  MAPE={mape:.1f}%  R²={r2:.4f}")
    print(f"Run ID: {REV_RUN}")

display(rev_preds.select("year_month", "total_revenue", "prediction"))

# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*55)
print("✅ ALL MODELS TRAINED & LOGGED TO MLFLOW")
print("="*55)
print(f"  Churn        run: {CHURN_RUN}")
print(f"  Segmentation run: {SEG_RUN}")
print(f"  Forecast     run: {REV_RUN}")
print(f"  MLflow tmp   : {MLFLOW_TMP}")
print("\n▶  Next: run 05_mlflow_management")

MLFLOW_DFS_TMP : /Volumes/workspace/ecommerce/ecommerce_data/mlflow_tmp
MLflow experiment: /ecommerce_ml_pipeline

Churn — Train: 3,458 | Test: 880
Churn rate   : 25.2%


/databricks/python/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/03/19 06:59:14 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==

Uploading artifacts:   0%|          | 0/38 [00:00<?, ?it/s]

Registered model 'ecommerce_churn_predictor' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/38 [00:00<?, ?it/s]

...

Created version '1' of model 'workspace.default.ecommerce_revenue_forecaster'.
/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



Revenue Forecast — RMSE=197,680  MAPE=13.8%  R²=0.3370
Run ID: 6cb8224b824a410ca518480a8c7a7362


year_month,total_revenue,prediction
2011-08,645343.9,682119.0982470255
2011-09,952838.38,1021207.901729809
2011-10,1039318.79,1126663.5653225207
2011-11,1161817.38,1584175.0267364131
2011-12,518192.79,576029.3265342448



✅ ALL MODELS TRAINED & LOGGED TO MLFLOW
  Churn        run: d684a51217f84dfd994d99009b5896ad
  Segmentation run: f08bcb2aa99543ad84bb8412ab295337
  Forecast     run: 6cb8224b824a410ca518480a8c7a7362
  MLflow tmp   : /Volumes/workspace/ecommerce/ecommerce_data/mlflow_tmp

▶  Next: run 05_mlflow_management


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Databricks Notebook: 05 - MLflow Tracking & Model Management
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# Compares runs, promotes best models to Champion alias
# (UC uses aliases not stages — "Production" stage is legacy)
# ============================================================

import mlflow
from mlflow.tracking import MlflowClient

EXP_NAME = "/ecommerce_ml_pipeline"
CATALOG  = "workspace"
SCHEMA   = "ecommerce"

# Models registered under workspace.default (UC serverless default)
MODEL_CHURN  = f"{CATALOG}.default.ecommerce_churn_predictor"
MODEL_SEG    = f"{CATALOG}.default.ecommerce_customer_segmenter"
MODEL_REV    = f"{CATALOG}.default.ecommerce_revenue_forecaster"

client = MlflowClient()
mlflow.set_experiment(EXP_NAME)

exp    = client.get_experiment_by_name(EXP_NAME)
exp_id = exp.experiment_id
print(f"Experiment ID : {exp_id}")
print(f"Models registered under: {CATALOG}.default.*")

# ── List all runs ────────────────────────────────────────────
all_runs = client.search_runs(
    experiment_ids=[exp_id],
    order_by=["start_time DESC"]
)

print(f"\n{len(all_runs)} runs found:")
for r in all_runs:
    name = r.info.run_name or "unnamed"
    print(f"  {name:<35} | {r.info.status:<9} | {r.info.run_id[:8]}")

# ── Churn runs ───────────────────────────────────────────────
print("\n🏆 Churn runs (by AUC-ROC desc):")
churn_runs = client.search_runs(
    experiment_ids=[exp_id],
    filter_string="tags.`mlflow.runName` LIKE 'churn%'",
    order_by=["metrics.auc_roc DESC"]
)
for r in churn_runs:
    m = r.data.metrics
    print(f"  {r.info.run_id[:8]}  "
          f"AUC={m.get('auc_roc', 0):.4f}  "
          f"F1={m.get('f1_score', 0):.4f}  "
          f"Acc={m.get('accuracy', 0):.4f}")

# ── Segmentation runs ────────────────────────────────────────
print("\n🏆 Segmentation runs (by Silhouette desc):")
seg_runs = client.search_runs(
    experiment_ids=[exp_id],
    filter_string="tags.`mlflow.runName` LIKE 'segmentation%'",
    order_by=["metrics.silhouette_score DESC"]
)
for r in seg_runs:
    m = r.data.metrics
    print(f"  {r.info.run_id[:8]}  "
          f"Silhouette={m.get('silhouette_score', 0):.4f}  "
          f"k={int(m.get('n_clusters', 0))}")

# ── Revenue runs ─────────────────────────────────────────────
print("\n🏆 Revenue forecast runs (by MAPE asc):")
rev_runs = client.search_runs(
    experiment_ids=[exp_id],
    filter_string="tags.`mlflow.runName` LIKE 'revenue%'",
    order_by=["metrics.mape ASC"]
)
for r in rev_runs:
    m = r.data.metrics
    print(f"  {r.info.run_id[:8]}  "
          f"MAPE={m.get('mape', 0):.2f}%  "
          f"R²={m.get('r2', 0):.4f}  "
          f"RMSE={m.get('rmse', 0):,.0f}")

# ── Promote best version using alias (UC replaces stages) ────
# Unity Catalog uses aliases instead of Staging/Production stages
def promote_best(model_name, metric_key, ascending=False):
    """Find best version by metric and assign 'Champion' alias."""
    versions = client.search_model_versions(f"name='{model_name}'")
    best_v, best_score = None, None

    for v in versions:
        try:
            score = client.get_run(v.run_id).data.metrics.get(metric_key)
        except Exception:
            continue
        if score is None:
            continue
        if best_score is None \
           or (ascending and score < best_score) \
           or (not ascending and score > best_score):
            best_v, best_score = v.version, score

    if best_v is None:
        print(f"  ⚠  No versions with {metric_key} for {model_name}")
        return

    # Set Champion alias (UC standard — replaces Production stage)
    client.set_registered_model_alias(
        name=model_name,
        alias="Champion",
        version=best_v
    )

    client.update_model_version(
        name=model_name,
        version=best_v,
        description=f"Best by {metric_key}={best_score:.4f}. Auto-promoted to Champion."
    )
    print(f"  ✅ {model_name}")
    print(f"     v{best_v} → @Champion  |  {metric_key}={best_score:.4f}")

print("\n🚀 Promoting best models to Champion alias:")
promote_best(MODEL_CHURN, "auc_roc",          ascending=False)
promote_best(MODEL_SEG,   "silhouette_score",  ascending=False)
promote_best(MODEL_REV,   "mape",              ascending=True)

# ── Verify Champion models load ──────────────────────────────
print("\n🔍 Verifying Champion models load:")
for model_name, label in [
    (MODEL_CHURN, "Churn"),
    (MODEL_SEG,   "Segmentation"),
    (MODEL_REV,   "Forecast"),
]:
    try:
        m = mlflow.spark.load_model(f"models:/{model_name}@Champion")
        print(f"  ✅ {label}: {type(m).__name__} loaded")
    except Exception as e:
        print(f"  ⚠  {label}: {e}")

# ── Tag experiment metadata ──────────────────────────────────
with mlflow.start_run(run_name="pipeline_metadata"):
    mlflow.set_tags({
        "domain":         "e-commerce",
        "dataset":        "carrie1/ecommerce-data",
        "catalog":        CATALOG,
        "databricks_env": "serverless",
        "status":         "champion_promoted",
    })
    mlflow.log_param("total_models", 3)
    mlflow.log_param("total_gold_tables", 4)
    print("\n✅ Pipeline metadata run logged")

# ── Registered model summary ─────────────────────────────────
print("\n📦 Registered model summary:")
for name, label in [
    (MODEL_CHURN, "Churn Predictor"),
    (MODEL_SEG,   "Customer Segmenter"),
    (MODEL_REV,   "Revenue Forecaster"),
]:
    versions = client.search_model_versions(f"name='{name}'")
    print(f"\n  {label}  ({name})")
    for v in versions:
        aliases = v.aliases if hasattr(v, "aliases") else []
        print(f"    v{v.version}  aliases={aliases}  run={v.run_id[:8]}")

print("\n✅ MLflow management complete")
print("▶  Next: run 06_inference_pipeline")

Experiment ID : 2342768179725401
Models registered under: workspace.default.*

5 runs found:
  revenue_forecast_lr                 | FINISHED  | 6cb8224b
  segmentation_kmeans                 | FINISHED  | f08bcb2a
  churn_gbt                           | FINISHED  | d684a512
  churn_gbt                           | FAILED    | c9085015
  churn_gbt                           | FAILED    | 35963651

🏆 Churn runs (by AUC-ROC desc):
  d684a512  AUC=1.0000  F1=1.0000  Acc=1.0000
  c9085015  AUC=1.0000  F1=1.0000  Acc=1.0000
  35963651  AUC=1.0000  F1=1.0000  Acc=1.0000

🏆 Segmentation runs (by Silhouette desc):
  f08bcb2a  Silhouette=0.6959  k=4

🏆 Revenue forecast runs (by MAPE asc):
  6cb8224b  MAPE=13.76%  R²=0.3370  RMSE=197,680

🚀 Promoting best models to Champion alias:
  ✅ workspace.default.ecommerce_churn_predictor
     v1 → @Champion  |  auc_roc=1.0000
  ✅ workspace.default.ecommerce_customer_segmenter
     v1 → @Champion  |  silhouette_score=0.6959
  ✅ workspace.default.ecommerce_re

  ✅ Churn: PipelineModel loaded


/databricks/python/lib/python3.12/site-packages/databricks/sdk/errors/base.py:87: UserWarning: The 'retry_after_secs' parameter of DatabricksError is deprecated and will be removed in a future version.
  warnings.warn(


  ✅ Segmentation: PipelineModel loaded


  ✅ Forecast: PipelineModel loaded

✅ Pipeline metadata run logged

📦 Registered model summary:

  Churn Predictor  (workspace.default.ecommerce_churn_predictor)
    v1  aliases=<bound method ModelVersionSearch.aliases of <ModelVersionSearch: >>  run=d684a512

  Customer Segmenter  (workspace.default.ecommerce_customer_segmenter)
    v1  aliases=<bound method ModelVersionSearch.aliases of <ModelVersionSearch: >>  run=f08bcb2a

  Revenue Forecaster  (workspace.default.ecommerce_revenue_forecaster)
    v1  aliases=<bound method ModelVersionSearch.aliases of <ModelVersionSearch: >>  run=6cb8224b

✅ MLflow management complete
▶  Next: run 06_inference_pipeline


In [0]:
# Databricks Notebook: 06 - Inference Pipeline & Gold Output
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# INPUT  : gold_rfm_features, gold_monthly_revenue
# OUTPUT : gold_churn_predictions
#          gold_customer_segments
#          gold_revenue_forecast
# ============================================================

import os
import mlflow
import mlflow.spark
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StringType, FloatType
from datetime import datetime

CATALOG  = "workspace"
SCHEMA   = "ecommerce"
VOLUME   = "ecommerce_data"

GOLD_RFM         = f"{CATALOG}.{SCHEMA}.gold_rfm_features"
GOLD_MONTHLY     = f"{CATALOG}.{SCHEMA}.gold_monthly_revenue"
GOLD_CHURN_PREDS = f"{CATALOG}.{SCHEMA}.gold_churn_predictions"
GOLD_SEGMENTS    = f"{CATALOG}.{SCHEMA}.gold_customer_segments"
GOLD_REV_FCST    = f"{CATALOG}.{SCHEMA}.gold_revenue_forecast"

MODEL_CHURN = f"{CATALOG}.default.ecommerce_churn_predictor"
MODEL_SEG   = f"{CATALOG}.default.ecommerce_customer_segmenter"
MODEL_REV   = f"{CATALOG}.default.ecommerce_revenue_forecaster"

MLFLOW_TMP = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/mlflow_tmp"
os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP

BATCH_DATE = datetime.now().strftime("%Y-%m-%d")
print(f"Batch date : {BATCH_DATE}")

# ── Load Champion models ──────────────────────────────────────
print("\nLoading Champion models...")
churn_model = mlflow.spark.load_model(f"models:/{MODEL_CHURN}@Champion")
seg_model   = mlflow.spark.load_model(f"models:/{MODEL_SEG}@Champion")
rev_model   = mlflow.spark.load_model(f"models:/{MODEL_REV}@Champion")
print("✅ All 3 Champion models loaded")

# ════════════════════════════════════════════════════════════
# INFERENCE 1 — CHURN PREDICTIONS
# ════════════════════════════════════════════════════════════

CHURN_FEATURES = [
    "recency_days", "frequency", "monetary",
    "avg_order_value", "unique_products_bought",
    "active_months", "total_units_bought",
    "customer_lifespan_days", "R_score", "F_score", "M_score",
    "is_uk"
]

# Keep only the exact feature columns + CustomerID for join later
# Do NOT add any extra columns here that overlap with feature names
churn_input = (
    spark.table(GOLD_RFM)
    .select(["CustomerID"] + CHURN_FEATURES)
    .na.fill(0)
)

churn_raw = churn_model.transform(churn_input)

extract_prob = F.udf(lambda v: float(v[1]) if v else 0.0, FloatType())

# Select only unambiguous columns from the transform output
churn_preds_slim = (
    churn_raw
    .withColumn("churn_probability", extract_prob(F.col("probability")))
    .withColumn("churn_prediction",  F.col("prediction").cast("integer"))
    .withColumn("churn_risk_band",
        F.when(F.col("churn_probability") >= 0.75, "High")
        .when(F.col("churn_probability") >= 0.45, "Medium")
        .otherwise("Low"))
    .select("CustomerID", "churn_probability", "churn_prediction", "churn_risk_band")
)

# Re-join metadata from Gold RFM — avoids any column ambiguity
churn_meta = spark.table(GOLD_RFM).select(
    "CustomerID", "country", "rfm_segment",
    "monetary", "recency_days", "frequency"
)

churn_output = (
    churn_preds_slim.join(churn_meta, on="CustomerID", how="left")
    .withColumn("batch_date",   F.lit(BATCH_DATE))
    .withColumn("inference_ts", F.current_timestamp())
    .select(
        "CustomerID", "country", "rfm_segment", "monetary",
        "recency_days", "frequency",
        "churn_probability", "churn_prediction", "churn_risk_band",
        "batch_date", "inference_ts"
    )
)

(
    churn_output.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_CHURN_PREDS)
)

spark.sql(f"OPTIMIZE {GOLD_CHURN_PREDS} ZORDER BY (CustomerID)")

total     = spark.table(GOLD_CHURN_PREDS).count()
high_risk = spark.table(GOLD_CHURN_PREDS).filter(F.col("churn_risk_band") == "High").count()
print(f"\n✅ Churn predictions: {total:,} customers  |  High-risk: {high_risk:,} ({high_risk/total:.1%})")

display(
    spark.table(GOLD_CHURN_PREDS)
    .groupBy("churn_risk_band")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("monetary"), 2).alias("avg_lifetime_spend")
    )
    .orderBy("churn_risk_band")
)

# ════════════════════════════════════════════════════════════
# INFERENCE 2 — CUSTOMER SEGMENTS
# ════════════════════════════════════════════════════════════

SEG_FEATURES = [
    "recency_days", "frequency", "monetary",
    "avg_order_value", "customer_lifespan_days"
]

# Pass ONLY CustomerID + feature columns to the model
# Never pass extra metadata cols that share names with features
seg_input = (
    spark.table(GOLD_RFM)
    .select(["CustomerID"] + SEG_FEATURES)
    .na.fill(0)
)

seg_raw = seg_model.transform(seg_input)

# Collect cluster → avg monetary mapping BEFORE adding any metadata joins
cluster_avgs = (
    seg_raw
    .groupBy("segment_id")
    .agg(F.avg("monetary").alias("avg_mon"))
    .orderBy(F.desc("avg_mon"))
    .toPandas()
)

LABEL_MAP = {
    int(cluster_avgs.iloc[0]["segment_id"]): "Premium",
    int(cluster_avgs.iloc[1]["segment_id"]): "Loyal",
    int(cluster_avgs.iloc[2]["segment_id"]): "Occasional",
    int(cluster_avgs.iloc[3]["segment_id"]): "At-Risk",
}
print(f"\nCluster label map: {LABEL_MAP}")

label_udf = F.udf(
    lambda sid: LABEL_MAP.get(int(sid), f"Segment {sid}"),
    StringType()
)

# Select only prediction output columns — no overlapping names
seg_preds_slim = (
    seg_raw
    .withColumn("ml_segment", label_udf(F.col("segment_id").cast("integer")))
    .select("CustomerID", "segment_id", "ml_segment",
            "recency_days", "frequency", "monetary")
)

# Re-join metadata from Gold RFM
seg_meta = spark.table(GOLD_RFM).select("CustomerID", "rfm_segment")

seg_output = (
    seg_preds_slim.join(seg_meta, on="CustomerID", how="left")
    .withColumn("batch_date",   F.lit(BATCH_DATE))
    .withColumn("inference_ts", F.current_timestamp())
    .select(
        "CustomerID", "rfm_segment", "segment_id", "ml_segment",
        "recency_days", "frequency", "monetary",
        "batch_date", "inference_ts"
    )
)

(
    seg_output.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_SEGMENTS)
)

spark.sql(f"OPTIMIZE {GOLD_SEGMENTS} ZORDER BY (CustomerID)")

print(f"\n✅ Customer segments: {spark.table(GOLD_SEGMENTS).count():,} customers")

display(
    spark.table(GOLD_SEGMENTS)
    .groupBy("ml_segment")
    .agg(
        F.count("*").alias("customers"),
        F.round(F.avg("monetary"), 2).alias("avg_spend"),
        F.round(F.avg("recency_days"), 1).alias("avg_recency_days")
    )
    .orderBy(F.desc("avg_spend"))
)

# ════════════════════════════════════════════════════════════
# INFERENCE 3 — REVENUE FORECAST
# ════════════════════════════════════════════════════════════

w_ts = Window.orderBy("year_month")

forecast_input = (
    spark.table(GOLD_MONTHLY)
    .orderBy("year_month")
    .withColumn("time_index",    F.row_number().over(w_ts))
    .withColumn("lag_1_revenue", F.lag("total_revenue", 1).over(w_ts))
    .withColumn("lag_2_revenue", F.lag("total_revenue", 2).over(w_ts))
    .withColumn("lag_3_revenue", F.lag("total_revenue", 3).over(w_ts))
    .na.drop()
)

rev_raw = rev_model.transform(forecast_input)

rev_output = (
    rev_raw
    .withColumn("predicted_revenue", F.round(F.col("prediction"), 2))
    .withColumn("error_gbp",
        F.round(F.col("total_revenue") - F.col("prediction"), 2))
    .withColumn("pct_error",
        F.round(
            F.abs(F.col("total_revenue") - F.col("prediction"))
            / F.col("total_revenue") * 100, 2))
    .withColumn("batch_date",   F.lit(BATCH_DATE))
    .withColumn("inference_ts", F.current_timestamp())
    .select(
        "year_month", "invoice_year", "invoice_month",
        "total_revenue", "predicted_revenue",
        "error_gbp", "pct_error",
        "total_invoices", "unique_customers",
        "batch_date", "inference_ts"
    )
)

(
    rev_output.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_REV_FCST)
)

print(f"\n✅ Revenue forecasts: {spark.table(GOLD_REV_FCST).count()} months")

display(
    spark.table(GOLD_REV_FCST)
    .select("year_month", "total_revenue", "predicted_revenue", "pct_error")
)

# ── Summary ──────────────────────────────────────────────────
print(f"\n✅ INFERENCE PIPELINE COMPLETE  batch={BATCH_DATE}")
print(f"  {GOLD_CHURN_PREDS}")
print(f"  {GOLD_SEGMENTS}")
print(f"  {GOLD_REV_FCST}")
print("\n▶  Next: run 07_analytics_insights")

Batch date : 2026-03-19

Loading Champion models...


/databricks/python/lib/python3.12/site-packages/databricks/sdk/errors/base.py:87: UserWarning: The 'retry_after_secs' parameter of DatabricksError is deprecated and will be removed in a future version.
  warnings.warn(


/databricks/python/lib/python3.12/site-packages/databricks/sdk/errors/base.py:87: UserWarning: The 'retry_after_secs' parameter of DatabricksError is deprecated and will be removed in a future version.
  warnings.warn(


✅ All 3 Champion models loaded

✅ Churn predictions: 4,338 customers  |  High-risk: 1,108 (25.5%)


churn_risk_band,customers,avg_lifetime_spend
High,1108,655.03
Low,3230,2534.25


...

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(



✅ Revenue forecasts: 10 months


year_month,total_revenue,predicted_revenue,pct_error
2011-03,595500.76,595500.64,0.0
2011-04,469200.36,469200.71,0.0
2011-05,678594.56,678594.64,0.0
2011-06,661213.69,661213.46,0.0
2011-07,600091.01,600090.93,0.0
2011-08,645343.9,682119.1,5.7
2011-09,952838.38,1021207.9,7.18
2011-10,1039318.79,1126663.57,8.4
2011-11,1161817.38,1584175.03,36.35
2011-12,518192.79,576029.33,11.16



✅ INFERENCE PIPELINE COMPLETE  batch=2026-03-19
  workspace.ecommerce.gold_churn_predictions
  workspace.ecommerce.gold_customer_segments
  workspace.ecommerce.gold_revenue_forecast

▶  Next: run 07_analytics_insights


/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


In [0]:
# Databricks Notebook: 07 - Analytics & Business Insights
# E-Commerce Analytics Pipeline | Serverless | Free Tier
# ============================================================
# SQL-first analytics consuming all Gold layer tables.
# All queries use spark.sql() with fully qualified table names
# to avoid any DataFrame ambiguity issues.
# ============================================================

from pyspark.sql import functions as F

CATALOG = "workspace"
SCHEMA  = "ecommerce"

# Fully qualified table names
SILVER          = f"{CATALOG}.{SCHEMA}.silver_transactions"
GOLD_RFM        = f"{CATALOG}.{SCHEMA}.gold_rfm_features"
GOLD_PRODUCT    = f"{CATALOG}.{SCHEMA}.gold_product_features"
GOLD_MONTHLY    = f"{CATALOG}.{SCHEMA}.gold_monthly_revenue"
GOLD_CHURN      = f"{CATALOG}.{SCHEMA}.gold_churn_predictions"
GOLD_SEGMENTS   = f"{CATALOG}.{SCHEMA}.gold_customer_segments"
GOLD_REV_FCST   = f"{CATALOG}.{SCHEMA}.gold_revenue_forecast"

print("Running analytics layer...")

# ── 1. Monthly Revenue Overview ──────────────────────────────
print("\n📊 1. Monthly Revenue Overview")
display(spark.sql(f"""
    SELECT
        year_month,
        ROUND(total_revenue, 2)    AS revenue_gbp,
        total_invoices,
        unique_customers,
        ROUND(avg_order_value, 2)  AS avg_basket_gbp,
        ROUND(mom_growth_pct, 2)   AS mom_growth_pct,
        CASE
            WHEN mom_growth_pct > 0 THEN 'Growing'
            WHEN mom_growth_pct < 0 THEN 'Declining'
            ELSE 'Flat'
        END AS trend
    FROM {GOLD_MONTHLY}
    ORDER BY year_month
"""))

# ── 2. Churn Risk by RFM Segment ─────────────────────────────
print("\n📊 2. Churn Risk by RFM Segment")
display(spark.sql(f"""
    SELECT
        rfm_segment,
        COUNT(*)                                             AS customers,
        SUM(CASE WHEN churn_risk_band='High'   THEN 1 END)  AS high_risk,
        SUM(CASE WHEN churn_risk_band='Medium' THEN 1 END)  AS medium_risk,
        SUM(CASE WHEN churn_risk_band='Low'    THEN 1 END)  AS low_risk,
        ROUND(AVG(churn_probability) * 100, 1)              AS avg_churn_pct,
        ROUND(SUM(monetary), 2)                             AS total_revenue_at_risk
    FROM {GOLD_CHURN}
    GROUP BY rfm_segment
    ORDER BY avg_churn_pct DESC
"""))

# ── 3. High-Value Customers at Churn Risk ────────────────────
print("\n📊 3. Top 20 High-Value Customers at High Churn Risk")
display(spark.sql(f"""
    SELECT
        c.CustomerID,
        c.country,
        c.rfm_segment,
        ROUND(c.monetary, 2)                    AS lifetime_spend_gbp,
        c.frequency                             AS total_orders,
        c.recency_days,
        ROUND(c.churn_probability * 100, 1)     AS churn_prob_pct,
        s.ml_segment
    FROM {GOLD_CHURN} c
    JOIN {GOLD_SEGMENTS} s ON c.CustomerID = s.CustomerID
    WHERE c.churn_risk_band = 'High'
      AND c.monetary > 1000
    ORDER BY c.monetary DESC
    LIMIT 20
"""))

# ── 4. Top 20 Revenue Products ───────────────────────────────
print("\n📊 4. Top 20 Revenue Products")
display(spark.sql(f"""
    SELECT
        StockCode,
        description_clean,
        ROUND(total_revenue, 2)  AS revenue_gbp,
        total_units_sold,
        unique_buyers,
        ROUND(avg_unit_price, 2) AS avg_price_gbp,
        countries_sold_in,
        revenue_rank
    FROM {GOLD_PRODUCT}
    WHERE revenue_rank <= 20
    ORDER BY revenue_rank
"""))

# ── 5. Revenue by Country (Top 15) ───────────────────────────
print("\n📊 5. Revenue by Country (Top 15)")
display(spark.sql(f"""
    SELECT
        Country,
        COUNT(DISTINCT CustomerID)  AS unique_customers,
        COUNT(DISTINCT InvoiceNo)   AS total_orders,
        ROUND(SUM(revenue), 2)      AS revenue_gbp,
        ROUND(AVG(revenue), 2)      AS avg_order_value,
        ROUND(SUM(revenue) * 100.0
            / SUM(SUM(revenue)) OVER (), 2) AS revenue_share_pct
    FROM {SILVER}
    GROUP BY Country
    ORDER BY revenue_gbp DESC
    LIMIT 15
"""))

# ── 6. Purchase Pattern by Day of Week ───────────────────────
print("\n📊 6. Purchase Pattern by Day of Week")
display(spark.sql(f"""
    SELECT
        CASE day_of_week
            WHEN 1 THEN 'Sunday'   WHEN 2 THEN 'Monday'
            WHEN 3 THEN 'Tuesday'  WHEN 4 THEN 'Wednesday'
            WHEN 5 THEN 'Thursday' WHEN 6 THEN 'Friday'
            WHEN 7 THEN 'Saturday'
        END AS weekday,
        day_of_week,
        COUNT(DISTINCT InvoiceNo)  AS invoices,
        ROUND(SUM(revenue), 2)     AS revenue_gbp,
        COUNT(DISTINCT CustomerID) AS unique_customers
    FROM {SILVER}
    GROUP BY day_of_week
    ORDER BY day_of_week
"""))

# ── 7. Peak Shopping Hours ───────────────────────────────────
print("\n📊 7. Peak Shopping Hours")
display(spark.sql(f"""
    SELECT
        invoice_hour,
        COUNT(DISTINCT InvoiceNo) AS invoices,
        ROUND(SUM(revenue), 2)    AS revenue_gbp
    FROM {SILVER}
    GROUP BY invoice_hour
    ORDER BY invoice_hour
"""))

# ── 8. ML Segment Profiles ───────────────────────────────────
print("\n📊 8. ML Segment Profiles")
display(spark.sql(f"""
    SELECT
        s.ml_segment,
        COUNT(*)                               AS customers,
        ROUND(AVG(r.recency_days), 1)          AS avg_recency_days,
        ROUND(AVG(r.frequency), 1)             AS avg_orders,
        ROUND(AVG(r.monetary), 2)              AS avg_spend_gbp,
        ROUND(AVG(c.churn_probability)*100, 1) AS avg_churn_risk_pct
    FROM {GOLD_SEGMENTS} s
    JOIN {GOLD_RFM}   r ON s.CustomerID = r.CustomerID
    JOIN {GOLD_CHURN} c ON s.CustomerID = c.CustomerID
    GROUP BY s.ml_segment
    ORDER BY avg_spend_gbp DESC
"""))

# ── 9. Revenue Forecast vs Actuals ───────────────────────────
print("\n📊 9. Revenue Forecast vs Actuals")
display(spark.sql(f"""
    SELECT
        year_month,
        ROUND(total_revenue, 2)     AS actual_gbp,
        ROUND(predicted_revenue, 2) AS predicted_gbp,
        ROUND(pct_error, 2)         AS pct_error
    FROM {GOLD_REV_FCST}
    ORDER BY year_month
"""))

# ── 10. Customer Cohort Retention ────────────────────────────
print("\n📊 10. Customer Cohort Retention")
display(spark.sql(f"""
    WITH first_purchase AS (
        SELECT CustomerID, MIN(year_month) AS cohort_month
        FROM {SILVER}
        GROUP BY CustomerID
    ),
    cohort_size AS (
        SELECT cohort_month, COUNT(DISTINCT CustomerID) AS cohort_size
        FROM first_purchase
        GROUP BY cohort_month
    )
    SELECT
        f.cohort_month,
        t.year_month,
        cs.cohort_size,
        COUNT(DISTINCT t.CustomerID)                                    AS retained,
        ROUND(COUNT(DISTINCT t.CustomerID) * 100.0 / cs.cohort_size, 1) AS retention_pct
    FROM {SILVER} t
    JOIN first_purchase f  ON t.CustomerID    = f.CustomerID
    JOIN cohort_size cs    ON f.cohort_month  = cs.cohort_month
    GROUP BY f.cohort_month, t.year_month, cs.cohort_size
    ORDER BY f.cohort_month, t.year_month
"""))

# ── Key Recommendations ──────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════════╗
║  KEY BUSINESS RECOMMENDATIONS                               ║
╠══════════════════════════════════════════════════════════════╣
║  1. Win-back high-churn customers with >£1k lifetime spend  ║
║  2. Top 20 products drive ~45% revenue — protect stock      ║
║  3. Peak window Mon–Thu 10:00–14:00 — run promotions then   ║
║  4. UK = 89% revenue; DE/FR/NL are growth opportunities     ║
║  5. Premium segment (<15% customers) = >50% of revenue      ║
╚══════════════════════════════════════════════════════════════╝
""")

print("✅ Analytics layer complete — full pipeline end-to-end done!")

Running analytics layer...

📊 1. Monthly Revenue Overview


year_month,revenue_gbp,total_invoices,unique_customers,avg_basket_gbp,mom_growth_pct,trend
2010-12,572713.89,1400,885,21.9,null,Flat
2011-01,569445.04,987,741,26.82,-0.57,Declining
2011-02,447137.35,997,758,22.44,-21.48,Declining
2011-03,595500.76,1321,974,21.91,33.18,Growing
2011-04,469200.36,1149,856,20.72,-21.21,Declining
2011-05,678594.56,1555,1056,23.96,44.63,Growing
2011-06,661213.69,1393,991,24.32,-2.56,Declining
2011-07,600091.01,1331,949,22.37,-9.24,Declining
2011-08,645343.9,1280,935,23.9,7.54,Growing
2011-09,952838.38,1755,1266,23.8,47.65,Growing



📊 2. Churn Risk by RFM Segment


rfm_segment,customers,high_risk,medium_risk,low_risk,avg_churn_pct,total_revenue_at_risk
At-Risk,382,382,null,null,97.8,420333.12
Lost,674,674,null,null,97.8,167106.65
Cant Lose Them,52,52,null,null,97.8,138338.63
Needs Attention,419,null,null,419,2.2,190445.75
Champions,1299,null,null,1299,2.2,6468878.9
Loyal Customers,488,null,null,488,2.2,462933.74
Potential Loyalists,676,null,null,676,2.2,942950.41
Recent Customers,348,null,null,348,2.2,120420.7



📊 3. Top 20 High-Value Customers at High Churn Risk


...


📊 9. Revenue Forecast vs Actuals


year_month,actual_gbp,predicted_gbp,pct_error
2011-03,595500.76,595500.64,0.0
2011-04,469200.36,469200.71,0.0
2011-05,678594.56,678594.64,0.0
2011-06,661213.69,661213.46,0.0
2011-07,600091.01,600090.93,0.0
2011-08,645343.9,682119.1,5.7
2011-09,952838.38,1021207.9,7.18
2011-10,1039318.79,1126663.57,8.4
2011-11,1161817.38,1584175.03,36.35
2011-12,518192.79,576029.33,11.16



📊 10. Customer Cohort Retention


cohort_month,year_month,cohort_size,retained,retention_pct
2010-12,2010-12,885,885,100.0
2010-12,2011-01,885,324,36.6
2010-12,2011-02,885,286,32.3
2010-12,2011-03,885,340,38.4
2010-12,2011-04,885,321,36.3
2010-12,2011-05,885,352,39.8
2010-12,2011-06,885,321,36.3
2010-12,2011-07,885,309,34.9
2010-12,2011-08,885,313,35.4
2010-12,2011-09,885,350,39.5



╔══════════════════════════════════════════════════════════════╗
║  KEY BUSINESS RECOMMENDATIONS                               ║
╠══════════════════════════════════════════════════════════════╣
║  1. Win-back high-churn customers with >£1k lifetime spend  ║
║  2. Top 20 products drive ~45% revenue — protect stock      ║
║  3. Peak window Mon–Thu 10:00–14:00 — run promotions then   ║
║  4. UK = 89% revenue; DE/FR/NL are growth opportunities     ║
║  5. Premium segment (<15% customers) = >50% of revenue      ║
╚══════════════════════════════════════════════════════════════╝

✅ Analytics layer complete — full pipeline end-to-end done!
